# 01 - Synthetic data generation and data quality
Fictional company **PetroNexa Energy**. Compares RAW (defective) files with PROCESSED files and reads the actual validation results.

All operational, production, financial, maintenance, inventory, sensor and HSE data in this project are synthetic/simulated and created for portfolio demonstration purposes. They do not represent actual operations of a real company.

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30); pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
from og_oip import config
NOTICE = config.SYNTHETIC_NOTICE
print(NOTICE)

All operational, production, financial, maintenance, inventory, sensor and HSE data in this project are synthetic/simulated and created for portfolio demonstration purposes. They do not represent actual operations of a real company.


In [2]:
import json
man = json.load(open(config.REFERENCE_DIR / "generation_manifest.json"))
pd.DataFrame({k: {"rows": v["rows"]} for k, v in man["tables"].items() if not k.startswith("_")}).T

,rows
dim_date,1096
dim_field,4
dim_well,60
dim_equipment,48
dim_supplier,8
dim_material,40
dim_warehouse,4
bridge_equipment_well,452
fact_production,58686
fact_sensor,105216


In [3]:
print("Total fact rows in clean generator truth:", f"{man['total_fact_rows']:,}")
raw_p = pd.read_csv(config.RAW_DIR / "fact_production.csv", low_memory=False); proc_p = pd.read_csv(config.PROCESSED_DIR / "fact_production.csv")
print("production rows raw / processed:", len(raw_p), len(proc_p))
raw_p.head()

Total fact rows in clean generator truth: 291,319
production rows raw / processed: 58862 58642


,date,well_id,operating_hours,downtime_hours,oil_production_bbl,gas_production_mcf,water_production_bbl,pressure_psi,temperature_c,water_cut_pct,potential_production_bbl,oil_uom
0,2022-01-01,WELL-0001,24.00,0.00,42.40,70.80,54.00,"2,355.70",73.40,56.02,43.60,bbl
1,2022-01-02,WELL-0001,24.00,0.00,42.50,65.80,57.20,"2,347.60",73.70,57.37,42.50,bbl
2,2022-01-03,WELL-0001,17.13,6.87,31.60,49.00,46.00,"2,327.80",73.40,59.28,44.20,bbl
3,2022-01-04,WELL-0001,24.00,0.00,41.90,66.00,61.20,"2,382.90",73.80,59.36,41.90,bbl
4,2022-01-05,WELL-0001,24.00,0.00,36.50,57.40,50.10,"2,367.90",73.00,57.85,38.20,bbl


## Injected defects (generator ground truth)

In [4]:
dl = json.load(open(config.REFERENCE_DIR / "defect_injection_log.json"))
pd.Series(dl, name="injected").to_frame()

,injected
production_nonstandard_date_format,8756
production_invalid_date,44
production_oil_reported_in_m3,253
production_missing_pressure_psi,293
production_missing_temperature_c,293
production_missing_water_cut_pct,176
production_missing_oil_production_bbl,58
production_oil_outliers_x10,58
production_negative_oil,29
production_downtime_gt_24,29


## Validation results: raw vs processed

In [5]:
r = pd.read_csv(config.TABLES_DIR / "dq_results_raw.csv"); p = pd.read_csv(config.TABLES_DIR / "dq_results_processed.csv")
print(f"RAW: {len(r)} checks, {(r.status!='PASS').sum()} failed, {r.failed_rows.sum():,} failed rows (summed over checks)")
print(f"PROCESSED: {len(p)} checks, {(p.status!='PASS').sum()} failed")
r[r.status != "PASS"].sort_values("failed_rows", ascending=False).head(15)[["table", "check", "column", "failed_rows"]]

RAW: 179 checks, 60 failed, 36,743 failed rows (summed over checks)
PROCESSED: 178 checks, 0 failed


,table,check,column,failed_rows
161,fact_inventory,date_format_consistency,date,13272
153,fact_production,date_format_consistency,date,8767
159,fact_operating_cost,date_format_consistency,date,4664
165,fact_sensor,date_format_consistency,timestamp,2045
47,fact_inventory,null_check,supplier_id,876
167,fact_operating_cost,domain_check,cost_category,644
157,fact_sales,date_format_consistency,date,623
49,fact_inventory,null_check,unit_cost,440
35,fact_sensor,null_check,temperature_c,421
36,fact_sensor,null_check,pressure_psi,421


In [6]:
log = json.load(open(config.PROCESSED_DIR / "cleaning_log.json"))
pd.DataFrame({t: {k: v for k, v in l.items() if k in ("rows_in", "rows_out", "rows_quarantined")} for t, l in log.items()}).T

,rows_in,rows_out,rows_quarantined
dim_supplier,14,8,0
dim_date,1096,1096,0
dim_field,4,4,0
dim_well,60,60,0
dim_equipment,48,48,0
dim_material,40,40,0
dim_warehouse,4,4,0
bridge_equipment_well,452,452,0
fact_maintenance_material,1192,1184,8
fact_purchase_order,1965,1965,0


Quarantined rows (unparsable dates, missing/unknown keys) are kept in `data/processed/quarantine/`; they are not repaired or silently dropped.